# Anomaly Detection — Isolation Forest, One-Class SVM & LOF

Anomaly detection identifies observations that differ substantially from normal behavior.
This notebook compares three unsupervised approaches on a synthetic two-feature dataset:

1. **Isolation Forest** — isolates unusual points with random partitions.
2. **One-Class SVM** — learns a boundary around the normal training data.
3. **Local Outlier Factor (LOF)** — flags points whose local density differs from their neighbors.

The models are trained **only on normal observations**. Ground-truth labels are used solely
for evaluation, not for fitting or threshold selection.

**Workflow:** generate data → visualize → scale → train → compare → interpret → run inference.

## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import IsolationForest
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    precision_recall_fscore_support, roc_curve,
    precision_recall_curve, ConfusionMatrixDisplay,
)

sns.set_style("whitegrid")
SEED = 42
rng = np.random.default_rng(SEED)
print("Libraries imported successfully.")

## 2. Create a Synthetic Dataset

Imagine two sensors monitoring equipment in two stable operating modes. Normal samples form
two clusters; anomalies are generated away from both modes. This gives us labeled examples
for reliable evaluation while keeping the training process unsupervised.

- `0` = normal
- `1` = anomaly

In [ ]:
N_NORMAL = 1_000
N_ANOMALIES = 80
centers = np.array([[-1.5, -1.0], [2.0, 2.0]])

normal_a = rng.normal(loc=centers[0], scale=[0.8, 0.6], size=(N_NORMAL // 2, 2))
normal_b = rng.normal(loc=centers[1], scale=[0.7, 0.9], size=(N_NORMAL // 2, 2))
X_normal = np.vstack([normal_a, normal_b])

# Rejection sampling keeps injected anomalies outside both normal operating regions.
anomaly_parts = []
while sum(len(part) for part in anomaly_parts) < N_ANOMALIES:
    candidates = rng.uniform(low=[-6.0, -5.0], high=[7.0, 8.0], size=(N_ANOMALIES, 2))
    distances = np.linalg.norm(candidates[:, None, :] - centers[None, :, :], axis=2)
    accepted = candidates[distances.min(axis=1) > 3.0]
    anomaly_parts.append(accepted)
X_anomaly = np.vstack(anomaly_parts)[:N_ANOMALIES]

# Split normal data first: no labeled anomalies enter model fitting.
X_train, X_test_normal = train_test_split(X_normal, test_size=0.30, random_state=SEED)
X_test = np.vstack([X_test_normal, X_anomaly])
y_test = np.array([0] * len(X_test_normal) + [1] * len(X_anomaly))

# Shuffle the evaluation set so normal and anomalous rows are interleaved.
shuffle = rng.permutation(len(X_test))
X_test, y_test = X_test[shuffle], y_test[shuffle]

feature_names = ["sensor_1", "sensor_2"]
train_df = pd.DataFrame(X_train, columns=feature_names)
test_df = pd.DataFrame(X_test, columns=feature_names)
test_df["label"] = y_test

print(f"Training set (normal only): {X_train.shape[0]} rows")
print(f"Test set: {len(X_test)} rows — {(y_test == 0).sum()} normal, {(y_test == 1).sum()} anomalies")
test_df.head()

## 3. Exploratory Visualization

The training data represent known normal behavior. The evaluation set adds unusual points
that the models have not seen during fitting.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharex=True, sharey=True)

axes[0].scatter(X_train[:, 0], X_train[:, 1], s=24, alpha=0.65, color="#2878B5")
axes[0].set_title("Training Data — Normal Only")
axes[0].set_xlabel("Sensor 1")
axes[0].set_ylabel("Sensor 2")

normal_mask = y_test == 0
axes[1].scatter(X_test[normal_mask, 0], X_test[normal_mask, 1], s=24, alpha=0.6,
                label="Normal", color="#2878B5")
axes[1].scatter(X_test[~normal_mask, 0], X_test[~normal_mask, 1], s=42, alpha=0.85,
                label="Anomaly", color="#D1495B", marker="X")
axes[1].set_title("Test Data — Normal + Anomalies")
axes[1].set_xlabel("Sensor 1")
axes[1].legend()

plt.tight_layout()
plt.show()

## 4. Feature Scaling

Fit the scaler on the training data only to avoid leaking information from the evaluation
set. Scaling is essential for distance- and kernel-based methods such as One-Class SVM and LOF.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training mean after scaling:", X_train_scaled.mean(axis=0).round(3))
print("Training standard deviation:", X_train_scaled.std(axis=0).round(3))

## 5. Train Three Unsupervised Detectors

`contamination` / `nu` control the expected proportion of unusual training observations and
therefore influence the decision threshold. Here they are set to 5% for all three models;
in a real project, tune this operating point using validation data and business costs.

In [ ]:
models = {
    "Isolation Forest": IsolationForest(
        n_estimators=300, contamination=0.05, random_state=SEED, n_jobs=-1
    ),
    "One-Class SVM": OneClassSVM(
        kernel="rbf", gamma="scale", nu=0.05
    ),
    "Local Outlier Factor": LocalOutlierFactor(
        n_neighbors=20, contamination=0.05, novelty=True
    ),
}

for name, model in models.items():
    model.fit(X_train_scaled)
    print(f"Trained: {name}")

## 6. Evaluate and Compare Models

All detectors follow scikit-learn's convention: `predict()` returns `-1` for an anomaly and
`1` for a normal point. We convert this to `1 = anomaly`, and negate `score_samples()` so
that **higher anomaly scores always mean more anomalous**.

- **ROC AUC** measures ranking quality across thresholds.
- **PR AUC** focuses on the positive (anomaly) class and is especially useful with imbalance.
- **Precision, recall, and F1** measure performance at each model's chosen decision threshold.

In [ ]:
predictions = {}
anomaly_scores = {}
results = []

for name, model in models.items():
    y_pred = (model.predict(X_test_scaled) == -1).astype(int)
    score = -model.score_samples(X_test_scaled)
    predictions[name] = y_pred
    anomaly_scores[name] = score

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_test, y_pred, average="binary", zero_division=0
    )
    results.append({
        "Model": name,
        "ROC AUC": roc_auc_score(y_test, score),
        "PR AUC": average_precision_score(y_test, score),
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
    })

results_df = pd.DataFrame(results).sort_values("PR AUC", ascending=False).reset_index(drop=True)
results_df.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = {"Isolation Forest": "#2878B5", "One-Class SVM": "#F28E2B", "Local Outlier Factor": "#59A14F"}

for name in models:
    fpr, tpr, _ = roc_curve(y_test, anomaly_scores[name])
    precision, recall, _ = precision_recall_curve(y_test, anomaly_scores[name])
    axes[0].plot(fpr, tpr, lw=2, color=colors[name], label=name)
    axes[1].plot(recall, precision, lw=2, color=colors[name], label=name)

axes[0].plot([0, 1], [0, 1], "--", color="gray", label="Random")
axes[0].set(title="ROC Curves", xlabel="False Positive Rate", ylabel="True Positive Rate")
axes[1].set(title="Precision–Recall Curves", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend()
plt.tight_layout()
plt.show()

## 7. Interpret the Best Model

Select the model with the highest PR AUC on this evaluation set. The confusion matrix shows
threshold-based errors, while the score distributions show how well normal and anomalous
observations are separated.

In [ ]:
best_model_name = results_df.loc[0, "Model"]
best_model = models[best_model_name]
best_predictions = predictions[best_model_name]
best_scores = anomaly_scores[best_model_name]
print(f"Best model by PR AUC: {best_model_name}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions, display_labels=["Normal", "Anomaly"],
    cmap="Blues", colorbar=False, ax=axes[0]
)
axes[0].set_title(f"{best_model_name} — Confusion Matrix")

sns.histplot(best_scores[y_test == 0], bins=25, stat="density", alpha=0.55,
             label="Normal", color="#2878B5", ax=axes[1])
sns.histplot(best_scores[y_test == 1], bins=25, stat="density", alpha=0.55,
             label="Anomaly", color="#D1495B", ax=axes[1])
axes[1].set(title="Anomaly Score Distribution", xlabel="Anomaly score (higher = more unusual)")
axes[1].legend()
plt.tight_layout()
plt.show()

In [ ]:
# Decision regions in standardized feature space.
x_min, x_max = X_test_scaled[:, 0].min() - 0.7, X_test_scaled[:, 0].max() + 0.7
y_min, y_max = X_test_scaled[:, 1].min() - 0.7, X_test_scaled[:, 1].max() + 0.7
grid_x, grid_y = np.meshgrid(
    np.linspace(x_min, x_max, 350), np.linspace(y_min, y_max, 350)
)
grid = np.c_[grid_x.ravel(), grid_y.ravel()]
grid_decision = best_model.decision_function(grid).reshape(grid_x.shape)

plt.figure(figsize=(8, 6))
plt.contourf(grid_x, grid_y, grid_decision, levels=24, cmap="RdYlBu", alpha=0.75)
plt.contour(grid_x, grid_y, grid_decision, levels=[0], colors="black", linewidths=1.5)
plt.scatter(X_test_scaled[y_test == 0, 0], X_test_scaled[y_test == 0, 1],
            s=18, alpha=0.55, color="#174A7E", label="Normal")
plt.scatter(X_test_scaled[y_test == 1, 0], X_test_scaled[y_test == 1, 1],
            s=45, marker="X", color="#8C1D40", label="Anomaly")
plt.title(f"{best_model_name} — Learned Decision Regions")
plt.xlabel("Sensor 1 (standardized)")
plt.ylabel("Sensor 2 (standardized)")
plt.colorbar(label="Decision value (negative = anomaly)")
plt.legend()
plt.tight_layout()
plt.show()

## 8. Inference on New Observations

The helper below applies the same scaler and selected model to new sensor readings. Replace
the example values with your own observations; the input order must match training.

In [ ]:
def detect_anomaly(sensor_1, sensor_2, model=best_model, fitted_scaler=scaler):
    """Return a prediction and anomaly score for one raw sensor reading."""
    features = np.array([[sensor_1, sensor_2]], dtype=float)
    features_scaled = fitted_scaler.transform(features)
    is_anomaly = model.predict(features_scaled)[0] == -1
    score = -model.score_samples(features_scaled)[0]
    return {
        "sensor_1": sensor_1,
        "sensor_2": sensor_2,
        "prediction": "Anomaly" if is_anomaly else "Normal",
        "anomaly_score": round(float(score), 3),
    }

examples = [
    detect_anomaly(-1.5, -1.0),  # typical of operating mode A
    detect_anomaly(2.0, 2.0),    # typical of operating mode B
    detect_anomaly(6.0, -4.0),   # far outside both modes
]
pd.DataFrame(examples)

## 9. Key Takeaways

- **Train on normal data:** no anomaly labels were used to fit these detectors.
- **Prevent leakage:** fit preprocessing only on the training partition.
- **Compare more than accuracy:** PR AUC, recall, and false positives matter when anomalies are rare.
- **Choose thresholds deliberately:** the expected contamination rate affects the precision–recall trade-off.
- **Validate on real data:** synthetic results demonstrate the workflow but do not guarantee production performance.

To adapt the notebook, replace `X_train` and `X_test` with same-ordered numeric features from
your own dataset. Retain labels for evaluation when available, but keep them out of model fitting.